### Imports

In [2]:
import argparse
import logging
import os
import random
import sys
import numpy as np
from PIL import Image
import torch
import torch.backends.cudnn as cudnn
import torch.nn as nn
from torch.utils.data import DataLoader
import torchvision.transforms as T
from tqdm import tqdm
import zarr

from pathlib import Path

# Get the current notebook directory
notebook_dir = os.getcwd()
project_root = Path(notebook_dir).parent.parent.parent  # go up three levels
print(project_root)

# Add to Python path
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

# from datasets.dataset_synapse import Synapse_dataset
# from utils import test_single_volume
project_root = Path.cwd().parent  # adjust if needed
sys.path.insert(0, str(project_root))
from networks.vit_seg_modeling import VisionTransformer as ViT_seg
from networks.vit_seg_modeling import CONFIGS as CONFIGS_ViT_seg

# Funke Lab Tools
import daisy
from funlib.persistence import Array, open_ds, prepare_ds
from funlib.geometry import Coordinate, Roi


c:\Users\Waluigi\Desktop\github_repos


c:\Users\Waluigi\anaconda3\envs\transunet_pred\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### FUNC

In [3]:
# utils
inp_transforms_rgb = T.Compose(
    [
        T.ToTensor(),
    ]
)

In [4]:
def model_prediction_rgb(
    mask: Array,
    s2_array: Array, # this is the 10x array (s0 = 40x, s1 = 20x, s2 = 10x etc.)
    # patch_size_final: Array, # TODO: voxel or world?
    model: torch.nn.Module,
    device: torch.device,
    task: str,
    pred_save_path: str = None,
    ):

    def process_block(block: daisy.Block):
        # in data slice
        inslices = s2_array._Array__slices(block.read_roi)
        # print(f"Original slices (C, H, W order): {inslices}")

        # inslices = (inslices[1], inslices[2], inslices[0]) # PIL expects (H, W, C)
        # img = Image.fromarray(s2_array[inslices])
        # print(f"Input image shape for pt preds: {s2_array[inslices].shape}")

        # Extract the patch in native format (C, H, W)
        img_hwc = s2_array[inslices]
        # print(f"Patch shape (H, W, C): {img_hwc.shape}")
        
        # Transpose to (H, W, C) for PIL
        # From (C, H, W) -> (H, W, C)
        # img_hwc = np.transpose(img_chw, (1, 2, 0))
        # print(f"Patch shape (H, W, C): {img_hwc.shape}")

        # apply normalization + tensor conversion
        input = inp_transforms_rgb(img_hwc).unsqueeze(0).to(device) # look into inp_transforms_rgb
        # print(f"Final input dimenstions {input.shape}")
        
        # model prediction
        with torch.no_grad():
            preds = model(input)                 # [1, C, H, W]
            preds = torch.argmax(preds, dim=1)   # choose class per pixel → [1, H, W]
            preds = preds.squeeze(0)             # remove batch dim → [H, W]
            preds = preds.cpu().numpy()          # move to CPU numpy
        mask[block.write_roi] = preds

    # model expects 224x224 pixels patches
    block_roi = Roi((0, 0), (224, 224)) * s2_array.voxel_size # convert from pixel to world units

    pred_task = daisy.Task(
        task,
        total_roi=s2_array.roi,
        read_roi=block_roi,  # (offset, shape), shape is read as world units
        write_roi=block_roi,
        read_write_conflict=False,
        num_workers=2,
        process_function=process_block,
    )
    daisy.run_blockwise(tasks=[pred_task], multiprocessing=False)

    print('predictions finished!')

    # # Save the mask after inference if path is provided
    # if pred_save_path:
    #     import zarr
    #     # Create a new zarr file with the mask data
    #     zarr.save_array(pred_save_path, mask.data)
    #     print(f"Mask saved to: {pred_save_path}")

    return

In [5]:
def load_zarr_level(zarr_path, roi_type='he', level=2):
    """
    Load a specific pyramid level from your SVS Zarr file.
    
    Args:
        zarr_path: Path to the zarr file/group
        level: Pyramid level (0=40x, 1=20x, 2=10x, 3=5x)
    
    Returns:
        funlib.persistence.Array
    """
    if roi_type == 'he':
        return open_ds(zarr_path / "raw" / f"s{level}")
    elif roi_type == 'xml':
        return open_ds(zarr_path / "labels" / f"s{level}")


In [6]:
def compile_transunet(model_path, # location of .pth file
                      vit_name='R50-ViT-B_16', 
                      img_size=224, 
                      n_skip=3, 
                      vit_patches_size=16, 
                      num_classes=3):
        """
        Initialize the TransUNet model for inference
        
        Args:
            model_path: Path to your .pth checkpoint file
            vit_name: Type of ViT model (default: 'R50-ViT-B_16')
            img_size: Input image size (default: 224)
            n_skip: Number of skip connections (default: 3)
            vit_patches_size: ViT patch size (default: 16)
            num_classes: Number of output classes (default: 3)
        """
        img_size = img_size
        num_classes = num_classes
        device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        
        print(f"Using device: {device}")
        print(f"Loading model from: {model_path}")
        
        # Load configuration
        config_vit = CONFIGS_ViT_seg[vit_name]
        config_vit.n_classes = num_classes
        config_vit.n_skip = n_skip
        config_vit.patches.size = (vit_patches_size, vit_patches_size)
        
        if vit_name.find('R50') != -1:
            config_vit.patches.grid = (int(img_size/vit_patches_size), 
                                           int(img_size/vit_patches_size))
        
        # Create model
        model = ViT_seg(config_vit, img_size=img_size, num_classes=num_classes)
        
        # Load weights
        checkpoint = torch.load(model_path, map_location=device)
        
        # Handle DataParallel checkpoints (remove 'module.' prefix if present)
        if list(checkpoint.keys())[0].startswith('module.'):
            checkpoint = {k.replace('module.', ''): v for k, v in checkpoint.items()}
        
        model.load_state_dict(checkpoint)
        model.to(device)
        model.eval()
        
        print("Model loaded successfully!")

        return model, device

### EXECUTE

#### Data Prep

In [7]:
# loading data from .zarr
he_zarr = Path(r"E:\PROJ_DCIS\0_TEMP_transunet_training_patch_extraction\2015003_H&E.zarr")
he_s2 = load_zarr_level(he_zarr, roi_type='he', level=2)

xml_zarr = Path(r"E:\PROJ_DCIS\0_TEMP_transunet_training_patch_extraction\2015003_H&E.zarr")
xml_s2 = load_zarr_level(xml_zarr, roi_type='xml', level=2)

print(f"ROI (world coordinates): {he_s2.roi}")
print(f"voxel size: {he_s2.voxel_size}")

ROI (world coordinates): [0:18915120, 0:17569440] (18915120, 17569440)
voxel size: (1008, 1008)


In [ ]:
# mask: Array
# s2_array: Array # this is the 10x array (s0 = 40x, s1 = 20x, s2 = 10x etc.)
# patch_size_final: Array
# model: torch.nn.Module
# device: torch.device
# task: str
# pred_save_path: str = None

In [8]:
pred_zarr = Path(r"E:\PROJ_DCIS\0_TEMP_transunet_training_patch_extraction\2015003_H&E_pred_4class.zarr")

# prepare mask (one channel, but same HxW as H&E)
mask_shape = (he_s2.shape[0], he_s2.shape[1])  # (H, W)
mask_s2 = prepare_ds(
    pred_zarr,
    shape=mask_shape,
    offset=he_s2.offset,
    voxel_size=he_s2.voxel_size,
    axis_names=['y', 'x'],
    units=he_s2.units,
    dtype=np.uint8,  # Assuming class labels
    mode="w",
)

# Check mask shape before running inference
print(f"Mask shape: {mask_s2.shape}")
print(f"Mask dtype: {mask_s2.dtype}")
print(f"Mask ROI: {mask_s2.roi}")

Mask shape: (18765, 17430)
Mask dtype: uint8
Mask ROI: [0:18915120, 0:17569440] (18915120, 17569440)


In [10]:
# Load the model
model_path = r"E:\PROJ_DCIS\model\TU_dcisHE224\TU_pretrain_R50-ViT-B_16_skip3_epo150_bs8_224_4class\epoch_149.pth"
model, device = compile_transunet(model_path, num_classes=4)

Using device: cuda
Loading model from: E:\PROJ_DCIS\model\TU_dcisHE224\TU_pretrain_R50-ViT-B_16_skip3_epo150_bs8_224_4class\epoch_149.pth


C:\Users\Waluigi\AppData\Local\Temp\ipykernel_26200\1926284271.py:39: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint = torch.load(model_path, map_location=device)


Model loaded successfully!


In [11]:
# define task
task = "inference_task" # identifier for daisy task
pred_save_path = r"E:\PROJ_DCIS\0_TEMP_transunet_training_patch_extraction\2015003_H&E_pred_4class.zarr"

#### Prediction

In [12]:
# run model
model_prediction_rgb(
    mask=mask_s2,
    s2_array=he_s2, # this is the 10x array (s0 = 40x, s1 = 20x, s2 = 10x etc.)
    # patch_size_final: Array, # TODO: voxel or world?
    model=model,
    device=device,
    task=task,
    pred_save_path=pred_save_path,
    )

inference_task ✔: 100%|██████████| 6391/6391 [05:13<00:00, 20.41blocks/s, ⧗=0, ▶=0, ✔=6391, ✗=0, ∅=0]


Execution Summary
-----------------

  Task inference_task:

    num blocks : 6391
    completed ✔: 6391 (skipped 0)
    failed    ✗: 0
    orphaned  ∅: 0

    all blocks processed successfully
predictions finished!


### INSPECTION

#### Load H&E into Napari

In [13]:
import napari
import dask.array as da
import numpy as np
from pathlib import Path

def load_image_with_s2_mask(image_path, mask_path):
    """Load image pyramid but only show mask at s2 level with proper colors"""
    
    viewer = napari.Viewer()
    image_path = Path(image_path)
    mask_path = Path(mask_path)
    
    # Load image pyramid
    for i in range(4):
        level_path = image_path / 'raw' / f's{i}'
        if level_path.exists():
            img_data = da.from_zarr(level_path)
            is_rgb = len(img_data.shape) == 3 and img_data.shape[-1] == 3
            scale = [2**i, 2**i] if is_rgb else [2**i] * len(img_data.shape)
            
            # Only show s0 by default, but keep others available
            viewer.add_image(
                img_data,
                name=f'Image s{i}',
                scale=scale,
                rgb=is_rgb,
                contrast_limits=[0, 255],
                visible=(i==0)  # Only show highest resolution by default
            )
    
    # Load mask (only at s2)
    mask_s2_path = mask_path
    if mask_s2_path.exists():
        mask_data = da.from_zarr(mask_s2_path)
        
        # Squeeze if needed
        if len(mask_data.shape) == 3 and mask_data.shape[-1] == 1:
            mask_data = mask_data.squeeze(axis=-1)
        
        # Check unique values in mask
        try:
            sample = mask_data[:100, :100].compute()
            unique_values = np.unique(sample)
            print(f"Mask unique values: {unique_values}")
        except:
            unique_values = [0, 1, 2]
            print(f"Using default values: {unique_values}")
        
        # Scale for s2 (4x downsampled from s0)
        scale = [4, 4]
        
        # For older napari versions, we need to set colors after creation
        # First add the labels layer without color parameter
        mask_layer = viewer.add_labels(
            mask_data,
            name='Mask s2 (10x)',
            scale=scale,
            visible=True,
            opacity=0.6
        )
        
        # Then set the color mapping
        # Create a colormap for the labels
        # In older napari, we can use the color property
        color_dict = {
            0: [0, 1, 0, 0],      # Transparent (RGBA)
            1: [1, 0, 0, 0.8],    # Red with 80% opacity
            2: [0, 0, 1, 0.8],    # Blue with 80% opacity
        }
        
        # Apply the color mapping
        # This sets the color for each label value
        try:
            # Method 1: Using color property (older napari)
            mask_layer.color = color_dict
            print("Applied colors via color property")
        except:
            try:
                # Method 2: Using colormap
                from napari.utils.colormaps import LabelColormap
                mask_layer.colormap = LabelColormap(color_dict=color_dict)
                print("Applied colors via LabelColormap")
            except:
                # Method 3: Manual color assignment
                print("Could not set colors automatically. Will use default colors.")
                print("You can manually set colors in the napari GUI:")
                print("  - Click on the mask layer")
                print("  - In the layer controls, adjust colors for each label")
        
        print(f"✅ Mask loaded at s2: {mask_data.shape} with scale {scale}")
        
    else:
        print(f"❌ No mask found at {mask_s2_path}")
    
    return viewer

In [15]:
# load H&E + mask (only at s2)
img_path = r"E:\PROJ_DCIS\0_TEMP_transunet_training_patch_extraction\2015003_H&E.zarr"
mask_path = r"E:\PROJ_DCIS\0_TEMP_transunet_training_patch_extraction\2015003_H&E_pred_4class.zarr"
viewer = load_image_with_s2_mask(
    img_path,
    mask_path
)
napari.run()

c:\Users\Waluigi\anaconda3\envs\transunet_pred\lib\site-packages\napari\_vispy\layers\scalar_field.py:197: UserWarning: data shape (75063, 69720, 3) exceeds GL_MAX_TEXTURE_SIZE 32768 in at least one axis and will be downsampled. Rendering is currently in 2D mode.
  warnings.warn(


Mask unique values: [0 1 3]
Applied colors via color property
✅ Mask loaded at s2: (18765, 17430) with scale [4, 4]


### [DEP]

In [ ]:
import napari
import dask.array as da
from pathlib import Path

def view_funlib_zarr_safe(zarr_path):
    """View funlib zarr in napari without loading into memory"""
    
    viewer = napari.Viewer()
    zarr_path = Path(zarr_path)
    
    # Open as dask arrays - these are lazy and don't load data
    for i in range(4):
        level_path = zarr_path / 'raw' / f's{i}'
        if level_path.exists():
            # Create dask array directly from zarr
            data = da.from_zarr(level_path)
            
            # Determine if it's RGB (last dimension is 3)
            is_rgb = len(data.shape) == 3 and data.shape[-1] == 3
            
            # For RGB data, we need scale for spatial dimensions only (y, x)
            # For non-RGB data, we need scale for all dimensions
            if is_rgb:
                # RGB data: shape is (y, x, c) - scale only y and x
                scale = [2**i, 2**i]  # Only 2 spatial dimensions
                print(f"RGB data: applying scale {scale} to spatial dimensions")
            else:
                # Non-RGB data: shape is (y, x) or (z, y, x) - scale all dimensions
                n_spatial_dims = len(data.shape)
                scale = [2**i] * n_spatial_dims
                print(f"Non-RGB data: applying scale {scale} to {n_spatial_dims} dimensions")
            
            # Add to viewer
            viewer.add_image(
                data,
                name=f'Level {i}',
                scale=scale,
                rgb=is_rgb,
                contrast_limits=[0, 255],
                visible=(i==0)  # Only show highest resolution by default
            )
            print(f"✅ Added level {i} with shape {data.shape}, RGB: {is_rgb}")
    
    return viewer

# Use it
mask_zarr = Path(r"E:\testing_transunet_training_patch_extraction\2015003_H&E_img.zarr")
viewer = view_funlib_zarr_safe(mask_zarr)
napari.run()